In [1]:
# Import libraries and set project root path
import pandas as pd
import numpy as np
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"
print("Base path:", BASE)

Base path: /Users/alexia/Documents/CASA/Dissertation


In [2]:
# Reload all datasets so this notebook runs independently without 01_data_loading.ipynb

# TS045 (car/van availability by household) — replaces TS007A entirely; driving-age
# population (Pᵢ) is no longer used anywhere in the demand formula
ts045_path = os.path.join(BASE, "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv")
ts045_raw = pd.read_csv(ts045_path, skiprows=7).rename(columns={
    "Area": "area_raw",
    "No cars or vans in household": "cars_0",
    "1 car or van in household": "cars_1",
    "2 cars or vans in household": "cars_2",
    "3 or more cars or vans in household": "cars_3plus",
})
ts045_raw = ts045_raw.dropna(subset=["area_raw"])
for col in ["cars_0", "cars_1", "cars_2", "cars_3plus"]:
    ts045_raw[col] = pd.to_numeric(ts045_raw[col], errors="coerce")
ts045_gor_london = ts045_raw[ts045_raw["area_raw"] == "gor:London"].copy()
ts045_lsoa = ts045_raw[ts045_raw["area_raw"].str.startswith("lsoa2021:", na=False)].copy()
parsed = ts045_lsoa["area_raw"].str.extract(r"lsoa2021:(?P<lsoa_code>\S+)\s*:\s*(?P<lsoa_name>.+)")
ts045_lsoa["lsoa_code"] = parsed["lsoa_code"]
ts045_lsoa["lsoa_name"] = parsed["lsoa_name"]
ts045 = ts045_lsoa[["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]].reset_index(drop=True)

# IoD2025 File 7
iod_raw = pd.read_csv(os.path.join(BASE, "03_data/demand/IoD2025.csv"))
cols_keep = [
    "LSOA code (2021)", "LSOA name (2021)",
    "Local Authority District code (2024)", "Local Authority District name (2024)",
    "Index of Multiple Deprivation (IMD) Score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
    "Income Score (rate)", "Income Rank (where 1 is most deprived)",
    "Income Decile (where 1 is most deprived 10% of LSOAs)",
]
iod2025 = iod_raw[cols_keep].rename(columns={
    "LSOA code (2021)": "lsoa_code", "LSOA name (2021)": "lsoa_name",
    "Local Authority District code (2024)": "lad_code",
    "Local Authority District name (2024)": "lad_name",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)": "income_score",
    "Income Rank (where 1 is most deprived)": "income_rank",
    "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
})

# OpenStreetEV GLA
osev_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/OpenStreetEV_GLA.csv"))
osev = osev_raw[["country_co","id","external_l","name","address","postal_cod","state",
                  "coordinate","coordina_1","location_c","location_1"]].rename(columns={
    "country_co": "country_code", "id": "location_id", "external_l": "external_uuid",
    "name": "location_name", "postal_cod": "postcode", "state": "borough",
    "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "location_1": "location_type",
})

# Zapmap (join_august2025)
zapmap_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/join_august2025.csv"), low_memory=False)
cols_keep = ["location_id","evse_id","address","city","state","coordinate","coordina_1",
             "location_c","zapmap_device_uid","power_band","device_max_power",
             "uid","charing_start","status","charging_duration","month","year","day"]
zapmap = zapmap_raw[cols_keep].rename(columns={
    "state": "borough", "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "charing_start": "charging_start", "uid": "session_uid",
})

print("All datasets reloaded successfully.")
print(f"ts045: {ts045.shape}, iod2025: {iod2025.shape}, osev: {osev.shape}, zapmap: {zapmap.shape}")

All datasets reloaded successfully.
ts045: (35672, 6), iod2025: (33755, 10), osev: (23045, 11), zapmap: (215037, 18)


In [3]:
# Create output directory for cleaned data if it does not exist
output_dir = os.path.join(BASE, "05_processed")
os.makedirs(output_dir, exist_ok=True)
print("Output directory:", output_dir)

Output directory: /Users/alexia/Documents/CASA/Dissertation/05_processed


## TS045: filter London + calculate Hᵢ and Cᵢ

In [4]:
# Define all 33 London boroughs (32 boroughs + City of London)
london_boroughs = [
    "City of London", "Barking and Dagenham", "Barnet", "Bexley", "Brent",
    "Bromley", "Camden", "Croydon", "Ealing", "Enfield", "Greenwich",
    "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow", "Havering",
    "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
    "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
    "Redbridge", "Richmond upon Thames", "Southwark", "Sutton",
    "Tower Hamlets", "Waltham Forest", "Wandsworth", "Westminster"
]

import re

def is_london(name, boroughs):
    if not isinstance(name, str):
        return False
    # Match borough name followed by space + digits (LSOA name format), avoids 'Brent' matching 'Brentwood'
    return any(re.match(rf"^{re.escape(b)}\s+\d", name) for b in boroughs)

ts045_london = ts045[ts045["lsoa_name"].apply(lambda x: is_london(x, london_boroughs))].reset_index(drop=True)

print("TS045 London rows:", len(ts045_london))

TS045 London rows: 4994


In [5]:
# Compute Hᵢ (households) and Cᵢ (average car/van ownership per household)
# Hᵢ = sum of the four TS045 bands (0, 1, 2, 3+ cars) — TS045's "Population: All households"
# means this sum is the household count directly, no separate household-count table needed.
# Cᵢ = weighted mean across the four bands, with "3 or more" approximated as 3 — a documented
# top-coding assumption (same disclosure-control pattern as the TS007A age-band split it replaces).
band_cols = ["cars_0", "cars_1", "cars_2", "cars_3plus"]
ts045_london["households"] = ts045_london[band_cols].sum(axis=1)
ts045_london["avg_car_ownership"] = (
    0 * ts045_london["cars_0"] + 1 * ts045_london["cars_1"]
    + 2 * ts045_london["cars_2"] + 3 * ts045_london["cars_3plus"]
) / ts045_london["households"]

census_london = ts045_london[["lsoa_code", "lsoa_name", "households", "avg_car_ownership"] + band_cols].copy()
census_london = census_london.rename(columns={"households": "Hi", "avg_car_ownership": "Ci"})

print("=== Census London (TS045-derived) ===")
print("Shape:", census_london.shape)
print(census_london.head(3).to_string())
print()
print("Hi (households) range:", census_london["Hi"].min(), "–", census_london["Hi"].max())
print("Ci (avg car ownership) range:", round(census_london["Ci"].min(), 3), "–", round(census_london["Ci"].max(), 3))

# Validate against the official gor:London aggregate row — total households should match closely
gor_total_households = ts045_gor_london[band_cols].sum(axis=1).values[0]
computed_total = census_london["Hi"].sum()
print()
print(f"Computed total households (4,994 LSOAs): {computed_total:,.0f}")
print(f"Official gor:London aggregate:           {gor_total_households:,.0f}")
print(f"Difference: {computed_total - gor_total_households:,.0f} ({(computed_total - gor_total_households) / gor_total_households:.4%})")

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/census_london_clean.csv")
census_london.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Census London (TS045-derived) ===
Shape: (4994, 8)
   lsoa_code            lsoa_name      Hi        Ci  cars_0  cars_1  cars_2  cars_3plus
0  E01000001  City of London 001A   837.0  0.395460     555   243.0    29.0        10.0
1  E01000002  City of London 001B   824.0  0.359223     578   208.0    26.0        12.0
2  E01000003  City of London 001C  1017.0  0.216323     826   169.0    15.0         7.0

Hi (households) range: 402.0 – 1304.0
Ci (avg car ownership) range: 0.11 – 1.985

Computed total households (4,994 LSOAs): 3,423,845
Official gor:London aggregate:           3,423,890
Difference: -45 (-0.0013%)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/census_london_clean.csv


## Task 1 — IoD2025 → London filter

### filter + match check

In [6]:
# Filter IoD2025 to Greater London LSOAs using the 2021 codes already confirmed in census_london.
# Both datasets are native LSOA 2021 boundaries, so unlike the old IMD2019 pipeline, no
# 2011→2021 lookup or imputation is needed — expect a clean 1:1 match.
london_codes = set(census_london["lsoa_code"])

iod2025_london = iod2025[iod2025["lsoa_code"].isin(london_codes)].reset_index(drop=True)

matched  = len(iod2025_london)
expected = len(london_codes)
missing  = london_codes - set(iod2025_london["lsoa_code"])

print("=== IoD2025 London ===")
print(f"Matched: {matched} / Expected: {expected}")
print(f"Missing LSOAs: {len(missing)}")
if missing:
    print("Sample missing codes:", list(missing)[:10])
print()
print(iod2025_london.head(3).to_string())

=== IoD2025 London ===
Matched: 4994 / Expected: 4994
Missing LSOAs: 0

   lsoa_code            lsoa_name   lad_code        lad_name  imd_score  imd_rank  imd_decile  income_score  income_rank  income_decile
0  E01000001  City of London 001A  E09000001  City of London      8.742     26525           8         0.013        33730             10
1  E01000002  City of London 001B  E09000001  City of London      4.722     31203          10         0.018        33669             10
2  E01000003  City of London 001C  E09000001  City of London      9.250     25913           8         0.107        25167              8


### validate value ranges

In [7]:
# Validate income_score range against proposal's documented range [0.002–0.998]
print("income_score stats:")
print(iod2025_london["income_score"].describe())
print()
print("income_decile value counts:")
print(iod2025_london["income_decile"].value_counts().sort_index())

income_score stats:
count    4994.000000
mean        0.270942
std         0.147900
min         0.010000
25%         0.152000
50%         0.262000
75%         0.372000
max         0.998000
Name: income_score, dtype: float64

income_decile value counts:
income_decile
1     513
2     837
3     838
4     676
5     502
6     412
7     338
8     290
9     265
10    323
Name: count, dtype: int64


### save imd_london_clean.csv

In [8]:
output_path = os.path.join(BASE, "05_processed/imd_london_clean.csv")
iod2025_london.to_csv(output_path, index=False)
print("Saved to:", output_path)
print("Shape:", iod2025_london.shape)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/imd_london_clean.csv
Shape: (4994, 10)


## Task 2 — Zapmap → datetime parsing + duration cleaning

### parse + filter

In [9]:
# Parse charging_start to datetime
zapmap["charging_start"] = pd.to_datetime(zapmap["charging_start"], errors="coerce")
parse_failures = zapmap["charging_start"].isna().sum()
print(f"charging_start parse failures: {parse_failures}")

# Drop rows with invalid charging_duration: <= 0 or > 10,080 minutes (7-day observation window)
rows_before = len(zapmap)
zapmap_clean = zapmap[
    (zapmap["charging_duration"] > 0) &
    (zapmap["charging_duration"] <= 10080) &
    (zapmap["charging_start"].notna())
].copy()
rows_after = len(zapmap_clean)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Dropped:     {rows_before - rows_after} ({(1 - rows_after/rows_before):.1%})")

charging_start parse failures: 0
Rows before: 215037
Rows after:  215037
Dropped:     0 (0.0%)


### add hour/dayofweek + save

In [10]:
# Add temporal columns for EDA (Block C in 03_EDA.ipynb)
zapmap_clean["hour"] = zapmap_clean["charging_start"].dt.hour
zapmap_clean["dayofweek"] = zapmap_clean["charging_start"].dt.dayofweek  # 0 = Monday

print("=== Zapmap Cleaned ===")
print("Shape:", zapmap_clean.shape)
print(zapmap_clean[["evse_id","charging_start","charging_duration","hour","dayofweek"]].head(5).to_string())
print()
print("charging_duration stats:")
print(zapmap_clean["charging_duration"].describe())
print()
print("Unique evse_id:", zapmap_clean["evse_id"].nunique())
print("Unique location_id:", zapmap_clean["location_id"].nunique())

output_path = os.path.join(BASE, "05_processed/zapmap_clean.csv")
zapmap_clean.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Zapmap Cleaned ===
Shape: (215037, 20)
                            evse_id      charging_start  charging_duration  hour  dayofweek
0  174b834d444d9c65929b489ce7e76fb6 2025-08-06 09:28:00                332     9          2
1  174b834d444d9c65929b489ce7e76fb6 2025-08-06 10:32:00                 88    10          2
2  174b834d444d9c65929b489ce7e76fb6 2025-08-06 10:39:00                952    10          2
3  174b834d444d9c65929b489ce7e76fb6 2025-08-06 18:12:00                235    18          2
4  174b834d444d9c65929b489ce7e76fb6 2025-08-07 06:16:00                119     6          3

charging_duration stats:
count    215037.000000
mean        416.071988
std         752.417130
min           1.000000
25%           7.000000
50%         180.000000
75%         448.000000
max        3600.000000
Name: charging_duration, dtype: float64

Unique evse_id: 10465
Unique location_id: 7582

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/zapmap_clean.csv


### compute ur_j per EVSE

In [11]:
# Compute ur_j per EVSE: total observed charging duration over the 7-day reference
# window (Aug 4–10, 2025), following the time-based definition in Borlaug et al. (2023).
# This is EVSE-level only — no spatial join to LSOA needed yet (that happens in 03_EDA
# once LSOA boundary geometries are loaded). ur_j feeds both Sᵢᵉᶠᶠ = Σur_j (supply-side,
# 03_EDA) and Uᵢ = 1 + mean(ur_j) (demand-side, 04_demand_estimation) from the same table.
evse_ur = (
    zapmap_clean.groupby("evse_id")["charging_duration"].sum()
    .reset_index()
    .rename(columns={"charging_duration": "total_duration_min"})
)
evse_ur["ur_j"] = evse_ur["total_duration_min"] / 10080

# Attach representative lat/lng + location_id per EVSE (first observed value; an EVSE's
# coordinates don't change within the 7-day window) — needed for the LSOA spatial join later
evse_coords = zapmap_clean.groupby("evse_id")[["location_id", "latitude", "longitude"]].first().reset_index()
evse_ur = evse_ur.merge(evse_coords, on="evse_id", how="left")

print("=== Per-EVSE ur_j (from observed sessions only) ===")
print("Unique EVSEs with ≥1 session:", len(evse_ur))
print(evse_ur["ur_j"].describe())
print()
print("Note: this covers only EVSEs with observed sessions in join_august2025.csv.")
print("EVSEs registered in OpenStreetEV_GLA with zero sessions (ur_j = 0) are added in")
print("03_EDA.ipynb once EVSEs are matched to LSOA via spatial join.")

output_path = os.path.join(BASE, "05_processed/evse_ur_clean.csv")
evse_ur.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Per-EVSE ur_j (from observed sessions only) ===
Unique EVSEs with ≥1 session: 10465
count    10465.000000
mean         0.848168
std          2.085984
min          0.000099
25%          0.000298
50%          0.342460
75%          0.988988
max         55.485714
Name: ur_j, dtype: float64

Note: this covers only EVSEs with observed sessions in join_august2025.csv.
EVSEs registered in OpenStreetEV_GLA with zero sessions (ur_j = 0) are added in
03_EDA.ipynb once EVSEs are matched to LSOA via spatial join.

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/evse_ur_clean.csv


## OpenStreetEV — duplicate + bounding box check

### check + clean + save

In [12]:
# Greater London approximate bounding box: lat 51.28–51.70, lon -0.51–0.33
print("=== OpenStreetEV Basic Checks ===")
print(f"Total rows: {len(osev)}")
print(f"Duplicate location_id: {osev.duplicated(subset=['location_id']).sum()}")
print(f"Missing latitude: {osev['latitude'].isna().sum()}")
print(f"Missing longitude: {osev['longitude'].isna().sum()}")

outside_bbox = osev[
    (osev["latitude"]  < 51.28) | (osev["latitude"]  > 51.70) |
    (osev["longitude"] < -0.51) | (osev["longitude"] > 0.33)
]
print(f"Rows outside London bounding box: {len(outside_bbox)}")

osev_clean = osev.drop_duplicates(subset=["location_id"], keep="first")
osev_clean = osev_clean[
    (osev_clean["latitude"]  >= 51.28) & (osev_clean["latitude"]  <= 51.70) &
    (osev_clean["longitude"] >= -0.51) & (osev_clean["longitude"] <= 0.33)
].reset_index(drop=True)

print()
print("=== OpenStreetEV Cleaned ===")
print("Shape:", osev_clean.shape)

output_path = os.path.join(BASE, "05_processed/osev_london_clean.csv")
osev_clean.to_csv(output_path, index=False)
print("Saved to:", output_path)

=== OpenStreetEV Basic Checks ===
Total rows: 23045
Duplicate location_id: 30
Missing latitude: 0
Missing longitude: 0
Rows outside London bounding box: 0

=== OpenStreetEV Cleaned ===
Shape: (23015, 11)
Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/osev_london_clean.csv


## Task 3 — Data Processing Summary Table

### pipeline_summary

In [13]:
# Data Processing Summary Table
# Note: eⱼ (existing EVSE count per LSOA), Sᵢᵉᶠᶠ, and Uᵢ all require a spatial join of
# EVSE/session coordinates to LSOA 2021 boundary geometries — not loaded in this notebook.
# That join happens in 03_EDA.ipynb; rows below marked "Pending" will be filled in there.
# Candidate-site / 50m-exclusion-buffer rows from the old OSM pipeline are removed — no
# longer part of the methodology (candidate nodes are now LSOA centroids, not OSM points).

summary_data = [
    ("LSOAs loaded (Greater London)",                  len(census_london)),
    ("Household total (Hi sum, validated vs gor:London)", int(census_london["Hi"].sum())),
    ("On-street EVSE locations (OpenStreetEV_GLA)",     int((osev_clean["location_category"] == "On-Street").sum())),
    ("Session records (join_august2025, post-clean)",   len(zapmap_clean)),
    ("Unique EVSEs with ≥1 session (ur_j computed)",    len(evse_ur)),
    ("EVSEs matched to LSOA via spatial join",          "Pending — needs LSOA boundary file (03_EDA)"),
    ("eⱼ / Sᵢᵉᶠᶠ / Uᵢ / has_evse_i (LSOA-level)",        "Pending — needs LSOA boundary file (03_EDA)"),
]

pipeline_summary = pd.DataFrame(summary_data, columns=["Item", "Count"])
print(pipeline_summary.to_string(index=False))

output_path = os.path.join(BASE, "05_processed/pipeline_summary.csv")
pipeline_summary.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

                                             Item                                       Count
                    LSOAs loaded (Greater London)                                        4994
Household total (Hi sum, validated vs gor:London)                                     3423845
      On-street EVSE locations (OpenStreetEV_GLA)                                       21366
    Session records (join_august2025, post-clean)                                      215037
     Unique EVSEs with ≥1 session (ur_j computed)                                       10465
           EVSEs matched to LSOA via spatial join Pending — needs LSOA boundary file (03_EDA)
        eⱼ / Sᵢᵉᶠᶠ / Uᵢ / has_evse_i (LSOA-level) Pending — needs LSOA boundary file (03_EDA)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/pipeline_summary.csv


## Final summary of all cleaned files

In [14]:
processed_dir = os.path.join(BASE, "05_processed")
files = [f for f in os.listdir(processed_dir) if f.endswith(".csv")]

print("=== Cleaned Files in 05_processed/ ===\n")
for f in sorted(files):
    path = os.path.join(processed_dir, f)
    df = pd.read_csv(path)
    print(f"{f}")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  Columns: {df.columns.tolist()}\n")

=== Cleaned Files in 05_processed/ ===

census_london_clean.csv
  Shape: 4,994 rows × 8 cols
  Columns: ['lsoa_code', 'lsoa_name', 'Hi', 'Ci', 'cars_0', 'cars_1', 'cars_2', 'cars_3plus']

evse_ur_clean.csv
  Shape: 10,465 rows × 6 cols
  Columns: ['evse_id', 'total_duration_min', 'ur_j', 'location_id', 'latitude', 'longitude']

imd_london_clean.csv
  Shape: 4,994 rows × 10 cols
  Columns: ['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'imd_score', 'imd_rank', 'imd_decile', 'income_score', 'income_rank', 'income_decile']

osev_london_clean.csv
  Shape: 23,015 rows × 11 cols
  Columns: ['country_code', 'location_id', 'external_uuid', 'location_name', 'address', 'postcode', 'borough', 'latitude', 'longitude', 'location_category', 'location_type']

pipeline_summary.csv
  Shape: 7 rows × 2 cols
  Columns: ['Item', 'Count']

zapmap_clean.csv
  Shape: 215,037 rows × 20 cols
  Columns: ['location_id', 'evse_id', 'address', 'city', 'borough', 'latitude', 'longitude', 'location_category', 'z